In [ ]:
import xarray as xr
import s3fs

# 1. Configuration & Auth
ENDPOINT = "http://192.168.1.237:9000"
BUCKET = "fcst-lib"

s3 = s3fs.S3FileSystem(
    key="d0d250b2541ac33f4660", 
    secret="2fb32d964768bc94a3c0", 
    client_kwargs={'endpoint_url': ENDPOINT}
)

# 2. Chunking Strategy
# Splits the 2GB file into chunks of ~70MB for efficient cloud reading
CHUNKS = {
    'time': 32,      # Keep time contiguous
    'member': 10,    # Split members into groups of 10
    'node': 54208    # Split space into 10 regions
}

def convert_forecasts():
    # Find all NC files in the 2025 folder
    input_pattern = f"{BUCKET}/forecasts/2025/*.nc"
    files = sorted(s3.glob(input_pattern))
    
    print(f"Found {len(files)} files. Starting conversion...")

    for i, nc_path in enumerate(files):
        # -- Path Logic --
        # Input:  fcst-lib/forecasts/2025/fcst_t2m_..._20250301.nc
        # Output: fcst-lib/zarr/2025/20250301.zarr
        
        filename = nc_path.split('/')[-1]
        # Extract date string (e.g., "20250301")
        date_str = filename.split('_')[-1].replace('.nc', '')
        
        # Define S3 destination
        zarr_path = f"s3://{BUCKET}/zarr/2025/{date_str}.zarr"

        print(f"[{i+1}/{len(files)}] Processing {date_str} -> {zarr_path}")

        # -- Conversion --
        try:
            # Open NetCDF (Lazily)
            with s3.open(nc_path, 'rb') as f:
                ds = xr.open_dataset(f, engine='h5netcdf')
                
                # Apply Chunking
                ds = ds.chunk(CHUNKS)

                # Save to Zarr
                # zarr_version=2 ensures standard compatibility and fixes warnings
                # consolidated=True creates the .zmetadata file for speed
                mapper = s3.get_mapper(zarr_path)
                ds.to_zarr(mapper, mode='w', consolidated=True, zarr_version=2)
                
        except Exception as e:
            print(f"!! Error on {filename}: {e}")

# Run
convert_forecasts()

Found 122 files. Starting conversion...
[1/122] Processing 20250301 -> s3://fcst-lib/zarr/2025/20250301.zarr


/tmp/ipykernel_2642354/2298567131.py:56: FutureWarning: zarr_version is deprecated, use zarr_format
  ds.to_zarr(mapper, mode='w', consolidated=True, zarr_version=2)


[2/122] Processing 20250302 -> s3://fcst-lib/zarr/2025/20250302.zarr


/tmp/ipykernel_2642354/2298567131.py:56: FutureWarning: zarr_version is deprecated, use zarr_format
  ds.to_zarr(mapper, mode='w', consolidated=True, zarr_version=2)


[3/122] Processing 20250303 -> s3://fcst-lib/zarr/2025/20250303.zarr


...

/tmp/ipykernel_2642354/2298567131.py:56: FutureWarning: zarr_version is deprecated, use zarr_format
  ds.to_zarr(mapper, mode='w', consolidated=True, zarr_version=2)


[9/122] Processing 20250309 -> s3://fcst-lib/zarr/2025/20250309.zarr


/tmp/ipykernel_2642354/2298567131.py:56: FutureWarning: zarr_version is deprecated, use zarr_format
  ds.to_zarr(mapper, mode='w', consolidated=True, zarr_version=2)


[10/122] Processing 20250310 -> s3://fcst-lib/zarr/2025/20250310.zarr


/tmp/ipykernel_2642354/2298567131.py:56: FutureWarning: zarr_version is deprecated, use zarr_format
  ds.to_zarr(mapper, mode='w', consolidated=True, zarr_version=2)
